# Aula 13 - PySpark: Funções de agregação

In [0]:
# Aula 13 - Funções de agregação (SUM, MAX, MIN + GROUP BY)
# Objetivo: resumir preços por modelo de carro

# Imports: functions (sum, max, min, regexp_replace) + types (DoubleType)
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Leitura do CSV com header e UTF-8; distinct() remove duplicatas na origem
df_carros = (
    spark.read
    .format('csv')
    .option('header', 'true')
    .option('encoding', 'utf-8')
    .load('/Workspace/Users/jcesarsantana215@gmail.com/Aulas PySpark/datasets/modelo_carro.csv', sep=',')
    .distinct()
)

# Normalização: remove o cifrão '$' e converte o preço para DOUBLE
# Sem isso, SUM/MAX/MIN tratariam o preço como texto (ordem alfabética, sem soma)
df_carros = df_carros.withColumn(
    'preco',
    regexp_replace(
        'preco', r'\$', '').cast(DoubleType())
)

display(df_carros)

In [0]:
# Agregação via SQL: soma, maior e menor preço por modelo
# GROUP BY modelo_carro + aliases (sum_preco, max_preco, min_preco) deixam o resultado legível
df_carros.createOrReplaceTempView('carros')

df_carros_sql = spark.sql(
    """
    SELECT
        modelo_carro, SUM(preco) AS sum_preco, MAX(preco) AS max_preco, MIN(preco) AS min_preco
    FROM carros
    GROUP BY modelo_carro
""")
display(df_carros_sql)

In [0]:
%sql
-- Mesma agregação com o magic %sql (prototipagem rápida no Databricks)
-- GROUP BY é obrigatório para colunas não agregadas (modelo_carro)
SELECT
    modelo_carro, SUM(preco) AS sum_preco, MAX(preco) AS max_preco, MIN(preco) AS min_preco
FROM carros
GROUP BY modelo_carro

In [0]:
# Agregação com a API PySpark: groupBy + agg
# .alias() renomeia cada métrica; o resultado equivale ao SQL acima
# Vantagem: encadeável com outros métodos (filter, orderBy, write...)
df_carros_spark = df_carros.groupby('modelo_carro').agg(
    sum('preco').alias('sum_preco'),
    max('preco').alias('max_preco'),
    min('preco').alias('min_preco')
)

display(df_carros_spark)